# SFT 概念与原理

Supervised Fine-Tuning（监督微调）——把预训练模型适配到对话任务。本节回答一个问题：**为什么 Wordle 任务需要 SFT？**

---

## 1. SFT 和预训练有什么区别？

```text
预训练（Pretraining）：
  输入："The capital of France is"
  输出："Paris."
  学的是：语言统计规律（什么词通常接在什么词后面）

监督微调（SFT）：
  输入：[user] "Round 1 feedback: X-X-X-X-X. Guess again."
  输出：[assistant] "<guess>SLIME</guess>"
  学的是：对话行为模式（用户问了什么、我该怎么回）
```

---

## 2. 为什么 Wordle 需要 SFT？

基座模型（如 Qwen3-1.7B）不知道什么是 Wordle。直接让它猜词，它不会按 `<guess>word</guess>` 格式输出——因为它从来没学过这个「规矩」。

SFT 做的事很简单：**用已有的 Wordle 游戏对话数据，教会模型两件事**——

- **格式规范**：输出必须带 `<guess>...</guess>` 标签。这是 SFT 最核心的目标——把格式正确率从 ~20% 提到 ~100%。
- **基本规则**：理解 G/Y/X 反馈的含义，学会利用反馈做简单推理。

至于「怎么在 6 轮内猜对」的策略优化——那是 RL 阶段的事。SFT 的角色是打好基础，让 RL 能在一个格式规范的起点上开始探索。

---

## 3. 一张表看清三阶段

| 维度 | Pretraining | SFT | RL |
|------|-------------|-----|-----|
| 学什么 | 语言规律 | 对话格式 | 策略优化 |
| 数据 | 任意文本 | 对话记录 | 模型自己探索 + 规则打分 |
| 信号 | 下一个 token | 标准回复 | 奖励分数 |
| 成本 | 极高 | 中 | 中-高 |

---

## 小结

- SFT 用对话数据教会模型「怎么回复」——格式规范 + 基本规则。
- 在 Wordle 两阶段方案中，SFT 是「脚手架」：打好格式基础，RL 才能有效探索。
- 下一章介绍训练 SFT 的框架——TorchTitan。

## 练习

1. （单选题）SFT 和 Pretraining 使用同一损失函数，主要区别在于？
    A. Pretraining 使用 MSE Loss，SFT 使用 CE Loss
    B. Pretraining 在所有 token 上计算 loss，SFT 只在 assistant 回复上计算 loss
    C. Pretraining 不需要训练数据，SFT 需要
    D. 两者完全相同

2. （判断题）在 Wordle SFT 中，user 消息（游戏反馈）被 mask（label = -100），因为模型的任务是学习生成 assistant 的回复，而非用户的反馈内容。

3. （多选题）SFT 在 Wordle 两阶段训练中扮演哪些角色？
    A. 格式对齐（学会 `<guess>word</guess>` 输出格式）
    B. 反馈理解（理解 G/Y/X 的含义）
    C. 最优策略（学会 6 轮内猜对的最优方法）
    D. 初步策略（基于反馈做基本推理）


In [ ]:
!cat ./answer/01.03_answer.txt